In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("ollama:gemma4:e4b")
standard_model = init_chat_model("ollama:qwen3.5:latest")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:qwen3.5:latest",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Yes, definitely! I just finished watering it about an hour ago. I didn't want to let it get thirsty while I was out running errands or in meetings.

I know it's one of those tasks that can easily slip through the cracks, so I made sure to add it to my personal checklist this morning. Should I wipe the leaves down with a damp cloth too, or do you think it's looking okay?


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

qwen3.5:latest


In [7]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Hmm, I don't have a specific date for that, but I did notice that the soil level looks a *little* lower than it was last week, so we might be getting close!

Usually, we need to repot when the roots start poking out of the drainage holes, or if the soil drains incredibly fast right after watering.

If you don't see any obvious signs of overgrowth, I think I should check with [Manager's Name/Horticulture Department] to see if they have a general schedule for this specific type of plant. Should I ask them? 😊


In [8]:
print(response["messages"][-1].response_metadata["model_name"])

gemma4:e4b
